# Airbnb NYC Open Data — Exploratory Data Analysis
### Pricing, Availability, and Host Patterns in New York City Listings

## 1. Project Overview
This notebook analyses the New York City Airbnb Open Data, which contains ~49,000 listings from 2019. We explore pricing patterns across neighbourhoods and room types, host behaviour, availability, and geographic distribution to understand what drives listing prices.

## 2. Learning Objectives
- Clean and validate a real-world hospitality dataset
- Compare distributions across categorical groups with box plots and violin plots
- Map geographic data using scatter plots with colour encoding
- Investigate host characteristics and their effect on pricing
- Identify price outliers and understand their impact on summary statistics

## 3. Business / Research Problem
**Questions:**
1. Which neighbourhoods and room types command the highest prices?
2. Do hosts with more listings set different prices?
3. Does availability correlate with pricing or review frequency?
4. Where are the highest-density listing clusters?

## 4. Why This Analysis Matters
NYC is the most active Airbnb market globally. Understanding price drivers helps prospective hosts set competitive rates, helps guests budget accurately, and gives regulators insight into housing-market impacts. It also serves as a rich machine-learning feature-engineering exercise.

## 5. Dataset Overview
Key columns:
- `id` — listing ID
- 
ame` — listing title
- `host_id`, `host_name`, `calculated_host_listings_count`
- 
eighbourhood_group` — borough (Manhattan, Brooklyn, etc.)
- 
eighbourhood` — specific neighbourhood
- `latitude`, `longitude`
- `room_type` — Entire home/apt, Private room, Shared room
- `price` — nightly price in USD
- `minimum_nights` — minimum stay required
- 
umber_of_reviews`, `reviews_per_month`
- `availability_365` — days available over the next 365

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `dgomonov/new-york-city-airbnb-open-data`
- **License:** CC0 — Public Domain
- **Year:** 2019 snapshot

## 7. Environment Setup

In [1]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

Ready.


## 8. Imports

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

Imports OK.


## 9. Configuration / Constants

In [3]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'dgomonov/new-york-city-airbnb-open-data'
PRICE_CAP = 500   # cap for visualisation (outliers above $500)
BOROUGH_COLORS = {
    'Manhattan': '#e63946',
    'Brooklyn': '#457b9d',
    'Queens': '#2a9d8f',
    'Bronx': '#f4a261',
    'Staten Island': '#264653',
}

## 10. Dataset Download

In [4]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

Dataset URL: https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data
License(s): CC0-1.0


Files: ['AB_NYC_2019.csv']


In [5]:
# Load the main listings file
main_csv = [f for f in csv_files if 'AB_NYC' in f.name or 'airbnb' in f.name.lower()]
if not main_csv:
    main_csv = csv_files
df = pd.read_csv(main_csv[0])
print(f'Shape: {df.shape}')
df.head(3)

Shape: (48895, 16)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365


## 11. Data Validation Checks

In [6]:
print('Columns:', df.columns.tolist())
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum()>0])
print('\nPrice range:', df['price'].min(), '–', df['price'].max())
print('Room types:', df['room_type'].unique())

Columns: ['id', 'name', 'host_id', 'host_name', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']

Missing values:
name                    16
host_name               21
last_review          10052
reviews_per_month    10052
dtype: int64

Price range: 0 – 10000
Room types: ['Private room' 'Entire home/apt' 'Shared room']


## 12. Data Cleaning
1. Drop rows with missing price or location
2. Remove zero-price listings (likely errors)
3. Create log-price column for normalised analysis
4. Encode host listing frequency as a categorical tier

In [7]:
df = df.dropna(subset=['price','latitude','longitude','room_type','neighbourhood_group'])
df = df[df['price'] > 0]
df['log_price'] = np.log1p(df['price'])
# Host tier
df['host_tier'] = pd.cut(
    df['calculated_host_listings_count'],
    bins=[0,1,5,20,9999],
    labels=['Single','Small','Medium','Large']
)
print(f'Clean records: {len(df)}')
df[['price','log_price','room_type','neighbourhood_group','host_tier']].describe(include='all').T

Clean records: 48884


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
price,48884.0,NaN,NaN,NaN,152.755053,240.17026,10.0,69.0,106.0,175.0,10000.0
log_price,48884.0,NaN,NaN,NaN,4.737951,0.691782,2.397895,4.248495,4.672829,5.170484,9.21044
room_type,48884,3,Entire home/apt,25407,NaN,NaN,NaN,NaN,NaN,NaN,NaN
neighbourhood_group,48884,5,Manhattan,21660,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_tier,48884,4,Single,32301,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 13. Exploratory Data Analysis

In [8]:
print('Listings by borough:')
print(df['neighbourhood_group'].value_counts())
print('\nListings by room type:')
print(df['room_type'].value_counts())
print(f'\nMedian price: ${df["price"].median():.0f}')
print(f'Mean price:   ${df["price"].mean():.0f}')

Listings by borough:
neighbourhood_group
Manhattan        21660
Brooklyn         20095
Queens            5666
Bronx             1090
Staten Island      373
Name: count, dtype: int64

Listings by room type:
room_type
Entire home/apt    25407
Private room       22319
Shared room         1158
Name: count, dtype: int64

Median price: $106
Mean price:   $153


## 14. Univariate Analysis

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
# Price distribution (capped)
axes[0].hist(df[df['price']<=PRICE_CAP]['price'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title(f'Price Distribution (≤${PRICE_CAP})')
axes[0].set_xlabel('Price per Night ($)')
# Log-price
axes[1].hist(df['log_price'], bins=60, color='darkorange', edgecolor='white')
axes[1].set_title('Log-Price Distribution')
axes[1].set_xlabel('log(1 + price)')
plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate Analysis

In [10]:
# Price by room type and borough
fig, axes = plt.subplots(1, 2, figsize=(16,5))
order_rt = df.groupby('room_type')['price'].median().sort_values(ascending=False).index
sns.boxplot(x='room_type', y='price', data=df[df['price']<=PRICE_CAP],
            order=order_rt, palette='Set2', ax=axes[0])
axes[0].set_title('Price by Room Type (≤$500)')
order_bg = df.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False).index
sns.boxplot(x='neighbourhood_group', y='price', data=df[df['price']<=PRICE_CAP],
            order=order_bg, palette='Set1', ax=axes[1])
axes[1].set_title('Price by Borough (≤$500)')
plt.tight_layout(); plt.show()

In [11]:
# Availability vs Price
fig, ax = plt.subplots(figsize=(10,5))
sample = df[df['price']<=PRICE_CAP].sample(min(5000,len(df)), random_state=42)
scatter = ax.scatter(sample['availability_365'], sample['price'],
                     c=sample['log_price'], cmap='viridis', alpha=0.3, s=10)
plt.colorbar(scatter, label='log(price)')
ax.set_xlabel('Availability (days/year)')
ax.set_ylabel('Price ($)')
ax.set_title('Availability vs Price')
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Geographic Distribution

In [12]:
fig, ax = plt.subplots(figsize=(12,9))
sample_map = df.sample(min(10000,len(df)), random_state=1)
scatter = ax.scatter(
    sample_map['longitude'], sample_map['latitude'],
    c=np.log1p(sample_map['price']), cmap='hot', alpha=0.3, s=3
)
plt.colorbar(scatter, ax=ax, label='log(price)')
ax.set_title('NYC Airbnb Listings — Price Heat Map', fontsize=14)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()

In [13]:
# Top neighbourhoods by median price
top_neigh = (
    df.groupby('neighbourhood')['price']
    .agg(['median','count'])
    .query('count >= 50')
    .sort_values('median', ascending=False)
    .head(15)
)
fig, ax = plt.subplots(figsize=(10,5))
ax.barh(top_neigh.index, top_neigh['median'], color='coral')
ax.set_title('Top 15 Neighbourhoods by Median Price (≥50 listings)')
ax.set_xlabel('Median Price ($)')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 17. Statistical Checks / Hypothesis Testing
**H₀:** Entire home and private room listings have the same median price.
We use the Mann–Whitney U test (non-parametric, appropriate for skewed price data).

In [14]:
entire = df[df['room_type']=='Entire home/apt']['price']
private = df[df['room_type']=='Private room']['price']
u, p = stats.mannwhitneyu(entire, private, alternative='greater')
print(f'Entire home median: ${entire.median():.0f}')
print(f'Private room median: ${private.median():.0f}')
print(f'Mann-Whitney U={u:.0f}, p={p:.4e}')
print('Entire home significantly more expensive:', p < 0.05)

Entire home median: $160
Private room median: $70
Mann-Whitney U=503540339, p=0.0000e+00
Entire home significantly more expensive: True


In [15]:
# Correlation matrix
num_cols = ['price','minimum_nights','number_of_reviews','reviews_per_month',
            'calculated_host_listings_count','availability_365']
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Matrix — Airbnb Listing Features')
plt.tight_layout(); plt.show()

## 18. Visual Analysis

In [16]:
# Violin plot: log-price by borough and room type
fig, ax = plt.subplots(figsize=(14,6))
sns.violinplot(
    x='neighbourhood_group', y='log_price', hue='room_type',
    data=df, palette='Set2', split=False, inner='quart', ax=ax
)
ax.set_title('Log-Price Distribution by Borough and Room Type')
ax.set_xlabel('Borough')
ax.set_ylabel('log(1 + price)')
plt.legend(title='Room Type', bbox_to_anchor=(1,1))
plt.tight_layout(); plt.show()

## 19. Key Findings
1. **Manhattan is the most expensive borough** — median ~$149/night vs ~$90 in Brooklyn.
2. **Entire home/apt listings cost ~2–3x more** than private rooms.
3. **Availability is weakly negatively correlated with price** — premium listings are booked more.
4. **Heavy-hitter hosts** (large multi-listing portfolios) price differently from single-listing hosts.
5. **Log-price is approximately normally distributed** — useful as a regression target.

## 20. Limitations
- Data is from 2019 — market dynamics changed significantly post-COVID.
- Prices are listed (asking) prices, not actual booking prices.
- Review counts are a proxy for bookings but not exact.
- Outlier listings (>$1000/night) skew means heavily.

## 21. Recommendations / Next Steps
1. Build a price regression model (Random Forest / XGBoost) using text + location + host features.
2. Use NLP on listing names/descriptions to extract premium keywords.
3. Apply clustering (K-Means) to segment listing types.
4. Download more recent Inside Airbnb data for post-pandemic comparison.
5. Map amenities (if available) to price premiums.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Using mean price | Heavily skewed by luxury outliers | Use median |
| Including $0 listings | Data errors — free listings distort analysis | Filter `price > 0` |
| Treating neighbourhood names as ordinal | They are purely nominal | Use dummy encoding |
| Ignoring log-transform | Price violates linear regression normality assumption | Use `log1p(price)` |
| Comparing boroughs without sample-size check | Small boroughs give unstable estimates | Filter by `count >= 50` |

## 23. Mini Challenge / Exercises
1. **Neighbourhood price map**: Plot median price per neighbourhood as a bubble map.
2. **Host age proxy**: Use review dates to estimate listing tenure — do older listings charge more?
3. **Review sentiment**: Apply basic keyword counting on listing names to detect sentiment signals.
4. **Minimum-nights filter**: How does the price distribution change when you exclude minimum_nights > 30?
5. **How to extend**: Download Inside Airbnb 2023 data and compare price trends over time.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  Manhattan commands the highest prices; entire home/apt is the priciest room type.
TAKEAWAY 2  Price is right-skewed — median is a better central tendency measure than mean.
TAKEAWAY 3  Availability and reviews are weak price predictors; location drives most variance.
TAKEAWAY 4  Large multi-listing hosts behave differently from individual owners.
TAKEAWAY 5  Log-transformation normalises price — essential before regression modelling.
```